<a href="https://colab.research.google.com/github/detektor777/colab_list_video/blob/main/colormnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }

!nvidia-smi

!git clone https://github.com/yyang181/colormnet.git
%cd colormnet

# ColorMNet dependencies
!git clone https://github.com/cheind/py-thin-plate-spline.git
!cd py-thin-plate-spline && pip install -e .

!pip install -r requirements.txt

# Устанавливаем ninja для корректной компиляции C++/CUDA расширений
!pip install ninja

!git clone https://github.com/ClementPinard/Pytorch-Correlation-extension.git

# Патч для исправления синтаксической ошибки OpenMP в PyTorch 2.x / Python 3.12
import os
cpp_file = "Pytorch-Correlation-extension/Correlation_Module/correlation.cpp"
if os.path.exists(cpp_file):
    with open(cpp_file, "r") as f:
        content = f.read()
    content = content.replace("#pragma omp parallel for", "// #pragma omp parallel for")
    with open(cpp_file, "w") as f:
        f.write(content)

# Установка скомпилированного модуля
!cd Pytorch-Correlation-extension && pip install .

!apt-get -y install ffmpeg

# Pretrained checkpoint
import os
os.makedirs('saves', exist_ok=True)
!wget -q -P ./saves https://github.com/yyang181/colormnet/releases/download/v0.1/DINOv2FeatureV6_LocalAtten_s2_154000.pth

In [ ]:
#@title ##**Select Video File** { display-mode: "form" }
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from google.colab import drive

upload_option = "Load from Google Drive Root"  #@param ["Upload from PC", "Load from Google Drive Root", "Load from Google Drive"]

file_name = None
last_selected_button = None

def reset_button_colors(buttons):
    for btn in buttons:
        btn.style.button_color = None

if upload_option == "Upload from PC":
    print("Please upload a video file.")
    uploaded = files.upload()
    if uploaded:
        file_name = list(uploaded.keys())[0]
    else:
        print("No file uploaded.")
        file_name = None

elif upload_option == "Load from Google Drive Root":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for f in os.listdir(root_dir):
        if os.path.isfile(os.path.join(root_dir, f)) and os.path.splitext(f)[1].lower() in video_extensions:
            files_list.append(f)

    if not files_list:
        print("No video files found in Google Drive root.")
        file_name = None
    else:
        print("Select a video file from Google Drive root:")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

elif upload_option == "Load from Google Drive":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            if os.path.splitext(f)[1].lower() in video_extensions:
                relative_path = os.path.relpath(os.path.join(dirpath, f), root_dir)
                files_list.append(relative_path)

    if not files_list:
        print("No video files found in Google Drive or its subfolders.")
        file_name = None
    else:
        print("Select a video file from Google Drive (including subfolders):")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

if file_name:
    print(f"Video file path set to: {file_name}")
else:
    print("Video file path not set. Please select a file.")


In [ ]:
#@title ##**Select Reference Image** { display-mode: "form" }
# This is the single color "exemplar" frame ColorMNet uses to propagate colors
# through the whole video. Pick a frame (or a similar-looking color photo) that
# matches the pose/lighting/composition of an early frame in your video for
# best results.
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from google.colab import drive

ref_upload_option = "Upload from PC"  #@param ["Upload from PC", "Load from Google Drive Root", "Load from Google Drive"]

ref_file_name = None
last_selected_ref_button = None

def reset_ref_button_colors(buttons):
    for btn in buttons:
        btn.style.button_color = None

image_extensions = ['.jpg', '.jpeg', '.png']

if ref_upload_option == "Upload from PC":
    print("Please upload the reference (exemplar) color image.")
    uploaded_ref = files.upload()
    if uploaded_ref:
        ref_file_name = list(uploaded_ref.keys())[0]
    else:
        print("No image uploaded.")
        ref_file_name = None

elif ref_upload_option == "Load from Google Drive Root":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    files_list = []
    for f in os.listdir(root_dir):
        if os.path.isfile(os.path.join(root_dir, f)) and os.path.splitext(f)[1].lower() in image_extensions:
            files_list.append(f)

    if not files_list:
        print("No image files found in Google Drive root.")
        ref_file_name = None
    else:
        print("Select a reference image from Google Drive root:")

        output = widgets.Output()
        buttons = []

        def on_ref_button_clicked(b):
            global ref_file_name, last_selected_ref_button
            with output:
                clear_output()
                reset_ref_button_colors(buttons)
                selected_file = b.description
                ref_file_name = os.path.join(root_dir, selected_file)

                if ref_file_name and os.path.exists(ref_file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_ref_button = b
                print(f"Selected file: {ref_file_name if ref_file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_ref_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

elif ref_upload_option == "Load from Google Drive":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    files_list = []
    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            if os.path.splitext(f)[1].lower() in image_extensions:
                relative_path = os.path.relpath(os.path.join(dirpath, f), root_dir)
                files_list.append(relative_path)

    if not files_list:
        print("No image files found in Google Drive or its subfolders.")
        ref_file_name = None
    else:
        print("Select a reference image from Google Drive (including subfolders):")

        output = widgets.Output()
        buttons = []

        def on_ref_button_clicked(b):
            global ref_file_name, last_selected_ref_button
            with output:
                clear_output()
                reset_ref_button_colors(buttons)
                selected_file = b.description
                ref_file_name = os.path.join(root_dir, selected_file)

                if ref_file_name and os.path.exists(ref_file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_ref_button = b
                print(f"Selected file: {ref_file_name if ref_file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_ref_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

if ref_file_name:
    print(f"Reference image path set to: {ref_file_name}")
else:
    print("Reference image path not set. Please select a file.")


In [ ]:
#@title ##**Config** { display-mode: "form" }
import os
import shutil
from google.colab import drive

first_frame_is_not_exemplar = True  #@param {type:"boolean"}
#@markdown Turn this ON if your reference image is a separate photo (not literally
#@markdown the first frame of the video). Leave it OFF only if the reference image
#@markdown you picked above *is* frame 0 of the video itself.

output_folder = "google_drive"  #@param ["google_drive", "root"]
video_name = ""  #@param {type:"string"}
#@markdown Leave blank to auto-derive the name from the video filename.

if not video_name:
    video_name = os.path.splitext(os.path.basename(file_name))[0]

if output_folder == "google_drive":
    if not os.path.exists('/content/drive'):
        print("Google Drive не подключён. Подключаем...")
        drive.mount('/content/drive')
    color_input_root = '/content/drive/MyDrive/color_input'
    color_output_root = '/content/drive/MyDrive/color_output'
elif output_folder == "root":
    color_input_root = '/content/color_input'
    color_output_root = '/content/color_output'

# Reference frames stay local — it's a single small image, no need for Drive I/O
color_ref_root = '/content/colormnet/tmp_ref'

color_input_folder = os.path.join(color_input_root, video_name)
color_output_folder = os.path.join(color_output_root, video_name)
color_ref_folder = os.path.join(color_ref_root, video_name)

#@markdown ---
clear_input_folder = True  #@param {type:"boolean"}
clear_output_folder = True  #@param {type:"boolean"}

if clear_input_folder and os.path.isdir(color_input_folder):
    shutil.rmtree(color_input_folder)
if clear_output_folder and os.path.isdir(color_output_folder):
    shutil.rmtree(color_output_folder)
# Если папка существует, но она пустая — удаляем её, чтобы test.py не считал её признаком завершенной работы
elif os.path.isdir(color_output_folder) and not os.listdir(color_output_folder):
    os.rmdir(color_output_folder)

if os.path.isdir(color_ref_folder):
    shutil.rmtree(color_ref_folder)

os.makedirs(color_input_folder, exist_ok=True)
# Внимание: мы НЕ создаем color_output_folder здесь. Скрипт test.py создаст её сам.
os.makedirs(color_ref_folder, exist_ok=True)

print(f"Video name:       {video_name}")
print(f"Input frames:      {color_input_folder}")
print(f"Reference frame:   {color_ref_folder}")
print(f"Output frames:     {color_output_folder}")

In [ ]:
#@title ##**Run sequence** { display-mode: "form" }
!pip install -q ffmpeg-python

import cv2
import imageio
import os
import shutil
import numpy as np
import tqdm
import time
import ffmpeg
from PIL import Image

library = "ffmpeg"  #@param ["cv2", "imageio", "ffmpeg"]
delay = "0.1"  #@param [0, 0.01, 0.05, 0.1, 0.2, 0.3]

# 8-digit zero padding avoids the lexicographic-sort bug that plain 3-digit
# padding hits on videos with 1000+ frames (test.py sorts frames by filename).
FRAME_NAME_DIGITS = 8

if library == "ffmpeg":
    probe = ffmpeg.probe(file_name)
    video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
    fps = video_info['r_frame_rate']
    duration = float(probe['format']['duration'])
    frame_count = int(video_info.get('nb_frames', 0)) or int(round(duration * eval(fps)))

    print("FPS:", fps)
    print("Duration:", duration)
    print("Frames (estimated):", frame_count)

    pbar_ffmpeg = tqdm.tqdm(total=frame_count, ncols=100, position=0, leave=True)
    process = (
        ffmpeg
        .input(file_name)
        .output('pipe:', format='rawvideo', pix_fmt='rgb24', qscale=0)
        .run_async(pipe_stdout=True)
    )

    i = 0
    frame_size = video_info['width'] * video_info['height'] * 3
    while True:
        raw_video = process.stdout.read(frame_size)
        if not raw_video or len(raw_video) < frame_size:
            break
        frame_path = f"{color_input_folder}/{i:0{FRAME_NAME_DIGITS}d}.png"
        if not os.path.isfile(frame_path):
            frame = np.frombuffer(raw_video, dtype='uint8').reshape((video_info['height'], video_info['width'], 3))
            imageio.imwrite(frame_path, frame)
        i += 1
        pbar_ffmpeg.update(1)
        if float(delay) > 0:
            time.sleep(float(delay))
    pbar_ffmpeg.close()
    process.wait()

elif library == "cv2":
    cap = cv2.VideoCapture(file_name)
    frame_count_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    i = 0
    with tqdm.tqdm(total=frame_count_total) as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_path = f"{color_input_folder}/{i:0{FRAME_NAME_DIGITS}d}.png"
            if not os.path.isfile(frame_path):
                cv2.imwrite(frame_path, frame)
            i += 1
            pbar.update(1)
    cap.release()

# Копирование референсного кадра с автоматическим ресайзом под видео для предотвращения падения модели
ref_ext = os.path.splitext(ref_file_name)[1].lower()
ref_target = os.path.join(color_ref_folder, f"{'0' * FRAME_NAME_DIGITS}{ref_ext}")

first_frame_path = os.path.join(color_input_folder, f"{0:0{FRAME_NAME_DIGITS}d}.png")
if os.path.exists(first_frame_path):
    try:
        video_img = Image.open(first_frame_path)
        video_w, video_h = video_img.size

        ref_img = Image.open(ref_file_name)
        ref_w, ref_h = ref_img.size

        if (ref_w, ref_h) != (video_w, video_h):
            print(f"Размеры референсного изображения ({ref_w}x{ref_h}) не совпадают с видео ({video_w}x{video_h}).")
            print(f"Автоматически изменяем размер референса до {video_w}x{video_h}...")
            ref_img_resized = ref_img.resize((video_w, video_h), Image.Resampling.LANCZOS)
            ref_img_resized.save(ref_target)
            print("Референс успешно оптимизирован и сохранен.")
        else:
            shutil.copy(ref_file_name, ref_target)
            print(f"Reference image copied to: {ref_target}")
    except Exception as e:
        print(f"Предупреждение при проверке размеров: {e}. Копируем оригинал.")
        shutil.copy(ref_file_name, ref_target)
else:
    shutil.copy(ref_file_name, ref_target)
    print(f"Reference image copied to: {ref_target}")

# Sanity check: make sure the extracted frame sequence has no gaps
def check_frames(frame_dir):
    frames = sorted(int(os.path.splitext(f)[0]) for f in os.listdir(frame_dir) if f.endswith('.png'))
    if not frames:
        print(f"No frames found in {frame_dir}!")
        return
    min_frame, max_frame = frames[0], frames[-1]
    missing = [i for i in range(min_frame, max_frame + 1) if i not in frames]
    print(f"{frame_dir}: {len(frames)} frames ({min_frame}..{max_frame})")
    if missing:
        print(f"Missing frames: {missing[:20]}{' ...' if len(missing) > 20 else ''}")
    else:
        print("All frames present")

check_frames(color_input_folder)

In [ ]:
#@title ##**Run Colorization** { display-mode: "form" }
import os
import subprocess
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

# Очищаем пустую папку вывода, если она существует
if os.path.exists(color_output_folder) and not os.listdir(color_output_folder):
    os.rmdir(color_output_folder)

cmd = [
    "python", "test.py",
    "--model", "saves/DINOv2FeatureV6_LocalAtten_s2_154000.pth",
    "--d16_batch_path", color_input_root,
    "--ref_path", color_ref_root,
    "--output", color_output_root,
]
if first_frame_is_not_exemplar:
    cmd.append("--FirstFrameIsNotExemplar")

print("Running:", " ".join(cmd))

# Запуск процесса с перенаправлением вывода для отображения в реальном времени
process = subprocess.Popen(
    cmd,
    cwd="/content/colormnet",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, # Объединяем стандартный вывод и ошибки
    text=True,
    bufsize=1
)

# Читаем лог выполнения в реальном времени
while True:
    line = process.stdout.readline()
    if not line and process.poll() is not None:
        break
    if line:
        print(line, end="")

process.wait()

if process.returncode != 0:
    raise RuntimeError(f"ColorMNet inference failed with return code {process.returncode}.")

def check_frames(frame_dir):
    if not os.path.isdir(frame_dir):
        print(f"{frame_dir} does not exist yet.")
        return
    frames = sorted(int(os.path.splitext(f)[0]) for f in os.listdir(frame_dir) if f.endswith('.png'))
    if not frames:
        print(f"No frames found in {frame_dir}!")
        return
    min_frame, max_frame = frames[0], frames[-1]
    missing = [i for i in range(min_frame, max_frame + 1) if i not in frames]
    print(f"{frame_dir}: {len(frames)} colorized frames ({min_frame}..{max_frame})")
    if missing:
        print(f"Missing frames: {missing[:20]}{' ...' if len(missing) > 20 else ''}")
    else:
        print("All frames colorized successfully")

check_frames(color_output_folder)

In [ ]:
#@title ##**Create video** { display-mode: "form" }
import cv2
import os
import subprocess
import time
from tqdm.notebook import tqdm
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

def log_time(start, message):
    elapsed = time.time() - start
    print(f"{message}: {elapsed:.2f} seconds")
    return time.time()

start_time = time.time()

base_file_name = os.path.basename(file_name)
output_file_name = os.path.splitext(base_file_name)[0] + '_colorized.mp4'

if output_folder == "google_drive":
    save_path = '/content/drive/MyDrive/'
else:
    save_path = '/content/'

output_file = os.path.join(save_path, output_file_name)
temp_video = "/content/temp_video.mp4"

start_time = log_time(start_time, "Initial setup")

cap = cv2.VideoCapture(file_name)
fps_of_video = cap.get(cv2.CAP_PROP_FPS) or 24
cap.release()

colorized_img_files = [os.path.join(color_output_folder, img) for img in os.listdir(color_output_folder) if img.lower().endswith('.png')]
colorized_img_files.sort()

if not colorized_img_files:
    raise ValueError(f"No colorized frames found in {color_output_folder}")

first_frame = cv2.imread(colorized_img_files[0], cv2.IMREAD_COLOR)
height, width = first_frame.shape[:2]
del first_frame

start_time = log_time(start_time, "Frame list preparation")

def get_video_bitrate(file_path):
    cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=bit_rate',
           '-of', 'default=noprint_wrappers=1:nokey=1', file_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    try:
        return int(result.stdout.strip())
    except ValueError:
        return None

bitrate = get_video_bitrate(file_name)
bitrate_str = f'{int(bitrate * 1.5) // 1000}k' if bitrate else '7500k'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_video, fourcc, fps_of_video, (width, height))

for img_file in tqdm(colorized_img_files, desc="Writing frames"):
    frame = cv2.imread(img_file, cv2.IMREAD_COLOR)
    if frame.shape[:2] != (height, width):
        frame = cv2.resize(frame, (width, height))
    out.write(frame)
    del frame

out.release()
gc.collect()

start_time = log_time(start_time, "Frame writing")

temp_converted = "/content/temp_converted.mp4"
cmd = ['ffmpeg', '-i', temp_video, '-c:v', 'libx264', '-b:v', bitrate_str, '-y', temp_converted]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    print(f"FFmpeg conversion failed: {result.stderr}")
    raise RuntimeError("Conversion to libx264 failed")
os.remove(temp_video)
os.rename(temp_converted, temp_video)

start_time = log_time(start_time, "FFmpeg conversion to libx264")

# Colorization only touches the picture — re-mux the ORIGINAL audio/subtitle
# tracks back onto the colorized video (the raw frame pipeline has no audio).
cmd = ['ffmpeg', '-i', temp_video, '-i', file_name, '-map', '0:v', '-map', '1:a?', '-map', '1:s?',
       '-c', 'copy', '-y', output_file]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    print(f"FFmpeg audio muxing failed: {result.stderr}")
    raise RuntimeError("Audio muxing failed")

start_time = log_time(start_time, "Audio/subtitle muxing")

if os.path.exists(output_file):
    if os.path.exists(temp_video):
        os.remove(temp_video)
    print("Video created successfully")
    print(f"Video saved at: {output_file}")
else:
    print("Failed to create video")
    print(f"FFmpeg error output: {result.stderr}")


In [ ]:
#@title ##**Download video** { display-mode: "form" }
from google.colab import files
import os
import time
from tqdm.notebook import tqdm

if not os.path.exists(output_file):
    raise FileNotFoundError(f"Video file {output_file} not found. Make sure the 'Create video' cell ran successfully.")

with tqdm(total=1, desc="Downloading video") as pbar:
    files.download(output_file)
    while os.path.exists(output_file) and os.path.getsize(output_file) == 0:
        time.sleep(0.5)
    pbar.update(1)

print("Download completed")
